In [199]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
#import pygwalker as pyg
from datetime import datetime
import os
#from ydata_profiling import ProfileReport
import csv
import seaborn as sns

In [208]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

## Imputing missing values ##

In [209]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [210]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [211]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


## Feature Engineering ##

In [212]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [223]:
df['Assortment'] = np.where(df['Assortment'] == 'b','b','other')

In [224]:
df.sample(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,PromoInterval,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,MonthStr,IsPromoMonth
78530,481,5,2015-05-22,6457,536,1,1,0,0,c,...,"Feb,May,Aug,Nov",2015,5,22,1,28.0,21,30.25,May,1
380151,860,4,2014-07-31,7291,822,1,1,0,1,c,...,NaN,2014,7,31,0,53.0,31,0.00,Jul,0
184198,224,1,2015-02-16,9475,610,1,1,0,0,d,...,"Jan,Apr,Jul,Oct",2015,2,16,1,25.0,8,25.75,Feb,0
593928,419,3,2014-01-15,4605,799,1,0,0,0,c,...,NaN,2014,1,15,0,52.0,3,0.00,Jan,0
645194,395,6,2013-11-30,3703,468,1,0,0,0,a,...,NaN,2013,11,30,0,9.0,48,0.00,Nov,0


In [225]:
df = df[df['Open'] == 1].copy()

In [226]:
df.shape

(844392, 27)

In [ ]:
df = df.sort_values('Date')

cutoff_date = '2015-06-01'

train = df[df['Date'] < cutoff_date]
valid = df[df['Date'] >= cutoff_date]

store_avg_sales = train.groupby('Store')['Sales'].mean().rename('Store_avg_sales')
store_avg_customers = train.groupby('Store')['Customers'].mean().rename('Store_avg_customers')
train = train.merge(store_avg_sales, on='Store', how='left')
train = train.merge(store_avg_customers, on='Store', how='left')
valid = valid.merge(store_avg_sales, on='Store', how='left')
valid = valid.merge(store_avg_customers, on='Store', how='left')

num_col=['CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths','Store_avg_sales','Store_avg_customers']
cat_col = ['Year','StoreType','Assortment']

X_train = train.drop(['Sales', 'Date'], axis=1)
y_train = train['Sales']

X_test = valid.drop(['Sales', 'Date'], axis=1)
y_test = valid['Sales']

(785781, 29)


In [230]:
X_train.drop(['Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)
X_test.drop(['Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [231]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first',sparse_output=True),cat_col)
],remainder='passthrough')

In [232]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [233]:
y_train_log = np.log1p(y_train)

In [236]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train_log)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    pred_log = model.predict(X_test)
    prediction_stop = time.perf_counter()

    Prediction = np.expm1(pred_log)
    #prediction = pred_log

    prediction_time_taken = prediction_stop-prediction_start
    # rmse = root_mean_squared_error(y_test,prediction)
    # rmsep = rmse/y_test_mean

    def rmspe(y_true, y_pred):
        mask = y_true != 0
        return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

    score = rmspe(y_test.values, Prediction)


    print(f'{model_name}: {score*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    # importance = pd.Series(model.feature_importances_, index=preprocessor.get_feature_names_out())
    # print(importance.sort_values(ascending=False).head(25))

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        #'rmse': rmse,
        'rmsep_percent': score * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

    errors = np.abs((y_test.values - Prediction) / y_test.values)
    error_df = valid.copy()
    error_df['prediction'] = Prediction
    error_df['pct_error'] = errors

    print(error_df.sort_values('pct_error', ascending=False).head(30)[
        ['Store','Date','Sales','prediction','pct_error','Promo','StateHoliday','StoreType','DayOfWeek']
    ])

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)



XGBRegressor: 16.28%

Training time taken for XGBRegressor: 7.9295

prediction time taken for XGBRegressor: 0.1171

       Store       Date  Sales    prediction  pct_error  Promo  StateHoliday  \
37465    292 2015-07-10   1012   5595.406250   4.529058      0             0   
24938    782 2015-06-26   1422   5470.867188   2.847305      0             0   
29033    909 2015-07-01   3547  13621.378906   2.840253      1             0   
2275     415 2015-06-03   2238   7361.745605   2.289431      1             0   
3913     415 2015-06-05   2427   7360.186523   2.032627      1             0   
31850    292 2015-07-04   1646   4664.416016   1.833789      0             0   
589      917 2015-06-01   3665  10174.599609   1.776153      1             0   
4926     415 2015-06-06   2389   6446.440918   1.698385      0             0   
3359     415 2015-06-04   1464   3766.252686   1.572577      1             1   
26783    909 2015-06-29   6125  15491.855469   1.529283      1             0   
2605